# Business Entity Resolution — End-to-End Notebook

Interactive version of the `code/business_entity_resolution/src/` pipeline:
**normalize -> block (candidate generation) -> compute pairwise features ->
train a LightGBM matcher -> tune a decision threshold -> evaluate (macro/micro
precision/recall/F1/F0.5) -> run inference on the test set -> write
`output/candidate_pairs.tsv` and `output/matching_results.tsv`.**

This notebook **imports the actual modules in `src/`** rather than duplicating
their code, and wraps the same logic `train.py` / `predict.py` use into three
reusable functions below. It always reflects the real pipeline: editing a
`.py` file and re-running a cell picks up the change immediately.

**Launch this notebook with your working directory set to
`code/business_entity_resolution/`** (its own folder) — the path config cell
below assumes that.

**How to use it:** run the *Setup*, *Config*, and *Core pipeline functions*
cells once, then jump to whichever of the **three run cells at the bottom**
you need:

1. **Smoke / dev run** — small S1 subsample, fast, writes to a separate
   `models_smoke/` so it never touches your real model. Use this to sanity
   check the pipeline before committing to the full run.
2. **Full train** — the real training run on the complete training set. This
   is the long one.
3. **Full test run + validation** — runs inference on the complete test set,
   writes both submission files, then runs `utils/validate_submission.py`.

## Setup

In [ ]:
import sys, os, json, time, subprocess
import numpy as np
import pandas as pd

SRC_DIR = os.path.join(os.getcwd(), "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import normalize
import blocking
import features as feat_mod
import metrics
import io_utils
import pipeline
import lightgbm as lgb

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


print("modules loaded OK from", SRC_DIR)


## Config — edit these for your run

In [ ]:
TRAIN_DIR = "../../dataset/train"
TEST_DIR = "../../dataset/test"
MODEL_DIR = "models"
OUTPUT_DIR = "../../output"

K_PER_SOURCE = 20   # candidates kept per source (S2, S3) per S1 entity after blocking
VAL_FRAC = 0.15
RANDOM_STATE = 42

assert os.path.isdir(TRAIN_DIR), (
    f"can't find {TRAIN_DIR} -- adjust TRAIN_DIR/TEST_DIR to match where you launched Jupyter from"
)
print("config OK, TRAIN_DIR ->", os.path.abspath(TRAIN_DIR))


## Core pipeline functions

Same logic as `src/train.py` and `src/predict.py`, refactored into callable
functions so the run cells at the bottom can invoke them with different
arguments (small subsample vs. full data) without duplicating any code.

In [ ]:
def train_model(train_dir, model_dir, k_per_source=20, max_s1=None,
                 val_frac=0.15, random_state=42):
    """Train the pairwise matcher end-to-end (mirrors train.py's main()).

    max_s1: subsample this many S1 training entities for a fast run;
    None trains on the full training set (the real competition run).
    Returns (booster, meta_dict).
    """
    os.makedirs(model_dir, exist_ok=True)

    log("loading train files ...")
    s1 = io_utils.load_source(os.path.join(train_dir, "train_source1.tsv"))
    s2 = io_utils.load_source(os.path.join(train_dir, "train_source2.tsv"))
    s3 = io_utils.load_source(os.path.join(train_dir, "train_source3.tsv"))
    gt = io_utils.load_ground_truth(os.path.join(train_dir, "train_ground_truth.tsv"))
    log(f"  s1={len(s1)} s2={len(s2)} s3={len(s3)} gt={len(gt)}")

    if max_s1:
        s1 = s1.sample(n=min(max_s1, len(s1)), random_state=random_state).reset_index(drop=True)
        gt = gt[gt["source1_entity_id"].isin(s1["entity_id"])].reset_index(drop=True)
        log(f"  subsampled to {len(s1)} S1 entities for this run")

    log("normalizing ...")
    s1n = pipeline.normalize_source(s1)
    s2n = pipeline.normalize_source(s2)
    s3n = pipeline.normalize_source(s3)

    all_s1_ids = s1n["entity_id"].tolist()
    rng = np.random.RandomState(random_state)
    shuffled = rng.permutation(all_s1_ids)
    n_val = int(len(shuffled) * val_frac)
    val_ids = set(shuffled[:n_val])
    train_ids = set(shuffled[n_val:])
    log(f"train S1={len(train_ids)} val S1={len(val_ids)}")

    log(f"blocking {len(s1n)} S1 entities against {len(s2n)} S2 / {len(s3n)} S3 records ...")
    t0 = time.time()
    pairs = pipeline.generate_candidates(s1n, s2n, s3n, k_per_source=k_per_source)
    log(f"  -> {len(pairs)} candidate pairs in {time.time() - t0:.1f}s")

    truth_by_s1 = {}
    for row in gt.itertuples(index=False):
        ids = row.matched_entity_ids.split(",") if row.matched_entity_ids else []
        truth_by_s1[row.source1_entity_id] = set(ids)

    log("computing features ...")
    t0 = time.time()
    feats = pipeline.compute_features_for_pairs(pairs, s1n, s2n, s3n)
    log(f"  -> features shape {feats.shape} in {time.time() - t0:.1f}s")
    feats["label"] = [
        int(cand in truth_by_s1.get(s1, ())) for s1, cand in zip(feats["s1_id"], feats["cand_id"])
    ]

    found_pairs_set = set(zip(feats["s1_id"], feats["cand_id"]))
    total_true = sum(len(truth_by_s1.get(s1, ())) for s1 in all_s1_ids)
    found_true = sum(
        1 for s1 in all_s1_ids for cand in truth_by_s1.get(s1, ()) if (s1, cand) in found_pairs_set
    )
    log(f"BLOCKING RECALL CEILING: {found_true}/{total_true} = "
        f"{(found_true / total_true if total_true else 1.0):.4f}")

    train_feats = feats[feats["s1_id"].isin(train_ids)].reset_index(drop=True)
    val_feats = feats[feats["s1_id"].isin(val_ids)].reset_index(drop=True)
    log(f"train pairs={len(train_feats)} (pos={train_feats['label'].sum()}) "
        f"val pairs={len(val_feats)} (pos={val_feats['label'].sum()})")

    X_train, y_train = train_feats[feat_mod.FEATURE_COLUMNS], train_feats["label"]
    X_val, y_val = val_feats[feat_mod.FEATURE_COLUMNS], val_feats["label"]

    n_pos, n_neg = y_train.sum(), len(y_train) - y_train.sum()
    scale_pos_weight = (n_neg / n_pos) if n_pos else 1.0
    log(f"scale_pos_weight={scale_pos_weight:.2f}")

    train_set = lgb.Dataset(X_train, label=y_train)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 30,
        "feature_fraction": 0.85,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        "seed": random_state,
    }

    log("training LightGBM ...")
    booster = lgb.train(
        params, train_set, num_boost_round=2000, valid_sets=[val_set],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )
    log(f"best iteration: {booster.best_iteration}")

    val_probs = booster.predict(X_val, num_iteration=booster.best_iteration)
    best_t, best_f = 0.5, -1.0
    for t in np.arange(0.05, 0.96, 0.02):
        pred_by_s1 = {}
        keep = val_probs >= t
        for s1, cand in zip(val_feats["s1_id"][keep], val_feats["cand_id"][keep]):
            pred_by_s1.setdefault(s1, set()).add(cand)
        f = metrics.macro_f_beta(pred_by_s1, truth_by_s1, list(val_ids), beta=0.5)
        if f > best_f:
            best_f, best_t = f, t
    log(f"BEST THRESHOLD={best_t:.2f}  VAL MACRO F0.5={best_f:.4f}")

    pred_by_s1 = {}
    keep = val_probs >= best_t
    for s1, cand in zip(val_feats["s1_id"][keep], val_feats["cand_id"][keep]):
        pred_by_s1.setdefault(s1, set()).add(cand)
    report = metrics.full_report(pred_by_s1, truth_by_s1, list(val_ids))
    log("VALIDATION METRICS @ tuned threshold:\n" + json.dumps(report, indent=2))

    # raw pairwise (row-level) precision/recall of the classifier itself, for
    # reference only -- NOT the same as the entity-level macro numbers above.
    pred_labels = (val_probs >= best_t).astype(int)
    tp = int(((pred_labels == 1) & (y_val == 1)).sum())
    fp = int(((pred_labels == 1) & (y_val == 0)).sum())
    fn = int(((pred_labels == 0) & (y_val == 1)).sum())
    pair_precision = tp / (tp + fp) if (tp + fp) else 0.0
    pair_recall = tp / (tp + fn) if (tp + fn) else 0.0
    log(f"pairwise (row-level) precision={pair_precision:.4f} recall={pair_recall:.4f} "
        f"(tp={tp} fp={fp} fn={fn})")

    importances = pd.Series(
        booster.feature_importance(importance_type="gain"), index=feat_mod.FEATURE_COLUMNS
    ).sort_values(ascending=False)
    log("top feature importances (gain):\n" + importances.head(15).to_string())

    model_path = os.path.join(model_dir, "matcher.txt")
    booster.save_model(model_path)
    meta = {
        "feature_columns": feat_mod.FEATURE_COLUMNS,
        "threshold": float(best_t),
        "val_metrics": report,
        "pairwise_precision": pair_precision,
        "pairwise_recall": pair_recall,
        "k_per_source": k_per_source,
        "best_iteration": booster.best_iteration,
    }
    with open(os.path.join(model_dir, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    log(f"saved model to {model_path} and meta.json")

    return booster, meta


In [ ]:
def _write_id_list(path, header_col, s1_to_ids, all_s1_ids):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(f"source1_entity_id\t{header_col}\n")
        for s1 in all_s1_ids:
            ids = s1_to_ids.get(s1, [])
            f.write(f"{s1}\t{','.join(ids)}\n")


def run_inference(test_dir, model_dir, output_dir, k_per_source_override=None,
                   threshold_override=None):
    """Run the trained model on a test set and write candidate_pairs.tsv +
    matching_results.tsv (mirrors predict.py's main()). Returns
    (candidate_by_s1, matched_by_s1)."""
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(model_dir, "meta.json"), encoding="utf-8") as f:
        meta = json.load(f)
    feature_columns = meta["feature_columns"]
    threshold = threshold_override if threshold_override is not None else meta["threshold"]
    k_per_source = k_per_source_override if k_per_source_override is not None else meta["k_per_source"]
    log(f"using threshold={threshold} k_per_source={k_per_source}")

    booster = lgb.Booster(model_file=os.path.join(model_dir, "matcher.txt"))

    log("loading test files ...")
    s1 = io_utils.load_source(os.path.join(test_dir, "test_source1.tsv"))
    s2 = io_utils.load_source(os.path.join(test_dir, "test_source2.tsv"))
    s3 = io_utils.load_source(os.path.join(test_dir, "test_source3.tsv"))
    log(f"  s1={len(s1)} s2={len(s2)} s3={len(s3)}")
    all_s1_ids = s1["entity_id"].tolist()

    log("normalizing ...")
    s1n = pipeline.normalize_source(s1)
    s2n = pipeline.normalize_source(s2)
    s3n = pipeline.normalize_source(s3)

    log("blocking (candidate generation) ...")
    t0 = time.time()
    pairs = pipeline.generate_candidates(s1n, s2n, s3n, k_per_source=k_per_source)
    log(f"  -> {len(pairs)} candidate pairs in {time.time() - t0:.1f}s")

    candidate_by_s1 = pairs.groupby("s1_id")["cand_id"].apply(list).to_dict()
    _write_id_list(os.path.join(output_dir, "candidate_pairs.tsv"), "candidate_entity_ids",
                    candidate_by_s1, all_s1_ids)
    log("wrote candidate_pairs.tsv")

    log("computing features ...")
    t0 = time.time()
    feats = pipeline.compute_features_for_pairs(pairs, s1n, s2n, s3n)
    log(f"  -> {feats.shape} in {time.time() - t0:.1f}s")

    log("scoring with model ...")
    feats["prob"] = booster.predict(feats[feature_columns])
    accepted = feats[feats["prob"] >= threshold]
    matched_by_s1 = accepted.groupby("s1_id")["cand_id"].apply(list).to_dict()
    _write_id_list(os.path.join(output_dir, "matching_results.tsv"), "matched_entity_ids",
                    matched_by_s1, all_s1_ids)
    log("wrote matching_results.tsv")

    n_with_match = sum(1 for v in matched_by_s1.values() if v)
    log(f"S1 entities with >=1 predicted match: {n_with_match} / {len(all_s1_ids)}")
    return candidate_by_s1, matched_by_s1


def validate_outputs(output_dir, test_dir, validate_script="../../utils/validate_submission.py"):
    """Run utils/validate_submission.py against the generated output files."""
    result = subprocess.run(
        [sys.executable, validate_script,
         "--matching", os.path.join(output_dir, "matching_results.tsv"),
         "--candidate", os.path.join(output_dir, "candidate_pairs.tsv"),
         "--test-dir", test_dir],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode


---
# Run cells

Run the ONE cell below that matches what you need right now. Each is
self-contained given the Setup/Config/functions above have already run.

## 1) Smoke / dev run
Fast sanity check on a small S1 subsample. Writes to `models_smoke/`, never touches your real model.

In [ ]:
booster, meta = train_model(
    TRAIN_DIR, model_dir="models_smoke",
    k_per_source=K_PER_SOURCE, max_s1=2000,
    val_frac=VAL_FRAC, random_state=RANDOM_STATE,
)


## 2) Full train
The real training run on the complete training set (no subsampling). This is the long one.

In [ ]:
booster, meta = train_model(
    TRAIN_DIR, model_dir=MODEL_DIR,
    k_per_source=K_PER_SOURCE, max_s1=None,
    val_frac=VAL_FRAC, random_state=RANDOM_STATE,
)


## 3) Full test run + validation
Runs inference on the complete test set, writes both submission files, then validates them.

In [ ]:
candidate_by_s1, matched_by_s1 = run_inference(TEST_DIR, MODEL_DIR, OUTPUT_DIR)
validate_outputs(OUTPUT_DIR, TEST_DIR)
